# Statistics for Decision Making
### HeroVired CPDA Graded Assignment
**Dataset:** property.csv — Australian Real Estate Prices  
**Author:** Mani Dixit &nbsp;|&nbsp; **Batch:** CPDA


## Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_1samp, ttest_ind, binom
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('property.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Shape: (13580, 23)
Columns: ['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG', 'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude', 'Longtitude', 'Regionname', 'Propertycount', 'Month', 'Year']


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Month,Year
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03,2.5,3067.0,...,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0,12,2016
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04,2.5,3067.0,...,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0,2,2016
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,3067.0,...,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0,3,2017


---
## Question 1: One-Sample T-Test — Altona Property Price
**Hypothesis:** Is the typical property price in Altona $800,000, or has it increased?  
**Significance level:** α = 0.05  
**Test:** One-sample t-test (one-tailed, right) — testing whether the population mean has risen above $800,000.


In [2]:
# ── Filter Altona ──────────────────────────────────────────────────────
altona = df[df['Suburb'] == 'Altona']['Price'].dropna()

print("=== Descriptive Statistics — Altona ===")
print(f"  n       : {len(altona)}")
print(f"  Mean    : ${altona.mean():,.2f}")
print(f"  Median  : ${altona.median():,.2f}")
print(f"  Std Dev : ${altona.std():,.2f}")

# ── Hypotheses ─────────────────────────────────────────────────────────
mu_0 = 800_000
print("\n=== Hypotheses ===")
print("  H0 (Null)       : mu = $800,000  (price has NOT changed)")
print("  H1 (Alternative): mu > $800,000  (price has INCREASED)")
print("  Test type       : One-tailed (right tail), alpha = 0.05")

# ── Test ───────────────────────────────────────────────────────────────
t_stat, p_two = ttest_1samp(altona, popmean=mu_0)
p_one = p_two / 2   # divide by 2 for one-tailed right

print("\n=== Test Results ===")
print(f"  T-statistic        : {t_stat:.4f}")
print(f"  Two-tailed p-value : {p_two:.4f}")
print(f"  One-tailed p-value : {p_one:.4f}")

# ── Decision ───────────────────────────────────────────────────────────
print("\n=== Decision (alpha = 0.05) ===")
if p_one < 0.05 and t_stat > 0:
    print("  REJECT H0")
    print("  Conclusion: There is statistically significant evidence that")
    print("  the typical Altona property price has INCREASED above $800,000.")
else:
    print("  FAIL TO REJECT H0")
    print(f"  p-value ({p_one:.4f}) > 0.05 — insufficient evidence to conclude")
    print("  the price has increased. The $800,000 benchmark remains plausible.")

=== Descriptive Statistics — Altona ===
  n       : 74
  Mean    : $834,830.41
  Median  : $773,500.00
  Std Dev : $291,546.05

=== Hypotheses ===
  H0 (Null)       : mu = $800,000  (price has NOT changed)
  H1 (Alternative): mu > $800,000  (price has INCREASED)
  Test type       : One-tailed (right tail), alpha = 0.05

=== Test Results ===
  T-statistic        : 1.0277
  Two-tailed p-value : 0.3075
  One-tailed p-value : 0.1537

=== Decision (alpha = 0.05) ===
  FAIL TO REJECT H0
  p-value (0.1537) > 0.05 — insufficient evidence to conclude
  the price has increased. The $800,000 benchmark remains plausible.


---
## Question 2: Two-Sample T-Test — Summer vs Winter Prices (2016)
**Hypothesis:** Is there any difference in property prices between summer and winter months in 2016?  
- **Winter months:** October–March (10, 11, 12, 1, 2, 3)  
- **Summer months:** April–September (4, 5, 6, 7, 8, 9)  
**Significance level:** α = 0.05  
**Test:** Welch's independent two-sample t-test (two-tailed — any difference)


In [3]:
# ── Filter 2016 & split by season ─────────────────────────────────────
df_2016 = df[df['Year'] == 2016].copy()

winter_months = [10, 11, 12, 1, 2, 3]
summer_months = [4, 5, 6, 7, 8, 9]

winter = df_2016[df_2016['Month'].isin(winter_months)]['Price'].dropna()
summer = df_2016[df_2016['Month'].isin(summer_months)]['Price'].dropna()

print("=== Descriptive Statistics — 2016 ===")
print(f"  Winter months (Oct-Mar): n={len(winter):,},  Mean=${winter.mean():,.2f},  Std=${winter.std():,.2f}")
print(f"  Summer months (Apr-Sep): n={len(summer):,},  Mean=${summer.mean():,.2f},  Std=${summer.std():,.2f}")

# ── Hypotheses ─────────────────────────────────────────────────────────
print("\n=== Hypotheses ===")
print("  H0 (Null)       : mu_winter = mu_summer  (no price difference between seasons)")
print("  H1 (Alternative): mu_winter != mu_summer (significant difference exists)")
print("  Test type       : Two-tailed, alpha = 0.05")
print("  Test chosen     : Welch's t-test (independent samples, variances not assumed equal)")

# ── Test ───────────────────────────────────────────────────────────────
t_stat, p_val = ttest_ind(winter, summer, equal_var=False)

print("\n=== Test Results ===")
print(f"  T-statistic : {t_stat:.4f}")
print(f"  P-value     : {p_val:.4f}")

# ── Decision ───────────────────────────────────────────────────────────
print("\n=== Decision (alpha = 0.05) ===")
if p_val < 0.05:
    print("  REJECT H0")
    print(f"  p-value ({p_val:.4f}) < 0.05 — statistically significant difference found.")
    print(f"  Winter mean (${winter.mean():,.0f}) > Summer mean (${summer.mean():,.0f}).")
    print("  Conclusion: Property prices in winter months are significantly higher")
    print("  than in summer months for 2016.")
else:
    print("  FAIL TO REJECT H0")
    print("  No statistically significant seasonal price difference in 2016.")

=== Descriptive Statistics — 2016 ===
  Winter months (Oct-Mar): n=2,300,  Mean=$1,116,647.59,  Std=$695,498.28
  Summer months (Apr-Sep): n=4,036,  Mean=$1,048,054.73,  Std=$621,493.44

=== Hypotheses ===
  H0 (Null)       : mu_winter = mu_summer  (no price difference between seasons)
  H1 (Alternative): mu_winter != mu_summer (significant difference exists)
  Test type       : Two-tailed, alpha = 0.05
  Test chosen     : Welch's t-test (independent samples, variances not assumed equal)

=== Test Results ===
  T-statistic : 3.9211
  P-value     : 0.0001

=== Decision (alpha = 0.05) ===
  REJECT H0
  p-value (0.0001) < 0.05 — statistically significant difference found.
  Winter mean ($1,116,648) > Summer mean ($1,048,055).
  Conclusion: Property prices in winter months are significantly higher
  than in summer months for 2016.


---
## Question 3: Binomial Probability — No Car Parking (Abbotsford)
**Question:** For the suburb of Abbotsford, what is the probability that out of 10 properties sold, exactly 3 will NOT have a car parking space?  
**Approach:** Estimate P(no car) from data, then apply Binomial distribution.


In [4]:
# ── Subset Abbotsford ─────────────────────────────────────────────────
abbotsford = df[df['Suburb'] == 'Abbotsford'].copy()
car_data   = abbotsford['Car'].dropna()

print("=== Car Parking Distribution — Abbotsford ===")
print(car_data.value_counts().sort_index().to_string())
print(f"\n  Total properties with car data : {len(car_data)}")

no_car_count = (car_data == 0).sum()
p_no_car     = no_car_count / len(car_data)
print(f"  Properties with NO car parking : {no_car_count}")
print(f"  P(no car parking)              : {p_no_car:.4f}  ({p_no_car*100:.2f}%)")

# ── Binomial Probability ───────────────────────────────────────────────
n, k = 10, 3
prob = binom.pmf(k, n, p_no_car)

print("\n=== Binomial Probability Calculation ===")
print(f"  Formula : P(X = k) = C(n,k) * p^k * (1-p)^(n-k)")
print(f"  n = {n}  (properties sold)")
print(f"  k = {k}  (properties with no car parking)")
print(f"  p = {p_no_car:.4f}  (P(no car) estimated from data)")
print(f"\n  P(exactly 3 out of 10 have NO car parking) = {prob:.3f}")

=== Car Parking Distribution — Abbotsford ===
Car
0.0    15
1.0    29
2.0    12

  Total properties with car data : 56
  Properties with NO car parking : 15
  P(no car parking)              : 0.2679  (26.79%)

=== Binomial Probability Calculation ===
  Formula : P(X = k) = C(n,k) * p^k * (1-p)^(n-k)
  n = 10  (properties sold)
  k = 3  (properties with no car parking)
  p = 0.2679  (P(no car) estimated from data)

  P(exactly 3 out of 10 have NO car parking) = 0.260


---
## Question 4: Probability of 3-Room Property (Abbotsford)
**Question:** In the suburb of Abbotsford, what are the chances of finding a property with 3 rooms?


In [5]:
rooms = abbotsford['Rooms']
total = len(rooms)
three_room_count = (rooms == 3).sum()
prob_3rooms = three_room_count / total

print("=== Room Distribution — Abbotsford ===")
print(rooms.value_counts().sort_index().to_string())
print(f"\n  Total Abbotsford properties : {total}")
print(f"  Properties with 3 rooms     : {three_room_count}")
print(f"\n  P(3 rooms) = {three_room_count}/{total} = {prob_3rooms:.3f}")

=== Room Distribution — Abbotsford ===
Rooms
1     5
2    27
3    20
4     4

  Total Abbotsford properties : 56
  Properties with 3 rooms     : 20

  P(3 rooms) = 20/56 = 0.357


---
## Question 5: Probability of 2-Bathroom Property (Abbotsford)
**Question:** In the suburb of Abbotsford, what are the chances of finding a property with 2 bathrooms?


In [6]:
bath = abbotsford['Bathroom'].dropna()
total_bath     = len(bath)
two_bath_count = (bath == 2).sum()
prob_2bath     = two_bath_count / total_bath

print("=== Bathroom Distribution — Abbotsford ===")
print(bath.value_counts().sort_index().to_string())
print(f"\n  Total Abbotsford properties (with bathroom data) : {total_bath}")
print(f"  Properties with 2 bathrooms                      : {two_bath_count}")
print(f"\n  P(2 bathrooms) = {two_bath_count}/{total_bath} = {prob_2bath:.3f}")

=== Bathroom Distribution — Abbotsford ===
Bathroom
1.0    35
2.0    19
3.0     2

  Total Abbotsford properties (with bathroom data) : 56
  Properties with 2 bathrooms                      : 19

  P(2 bathrooms) = 19/56 = 0.339


---
## Question 6: One-Sample T-Test — Richmond Industry Pricing Claim
**Claim:** A real estate firm claims the average property price in Richmond is $1,000,000.  
**Task:** Test whether the actual average price is significantly different from this claim at α = 0.05.


In [7]:
# ── Filter Richmond ────────────────────────────────────────────────────
richmond = df[df['Suburb'] == 'Richmond']['Price'].dropna()

print("=== Descriptive Statistics — Richmond ===")
print(f"  n       : {len(richmond)}")
print(f"  Mean    : ${richmond.mean():,.2f}")
print(f"  Median  : ${richmond.median():,.2f}")
print(f"  Std Dev : ${richmond.std():,.2f}")

# ── Null & Alternative Hypotheses ─────────────────────────────────────
mu_0 = 1_000_000
print("\n=== Null and Alternative Hypotheses ===")
print("  H0 (Null)       : mu = $1,000,000  (firm's claim is correct)")
print("  H1 (Alternative): mu != $1,000,000  (actual mean is significantly different)")
print("  Test type       : Two-tailed, alpha = 0.05")

# ── Test Statistic & P-Value ──────────────────────────────────────────
t_stat, p_val = ttest_1samp(richmond, popmean=mu_0)

print("\n=== Test Statistic & P-Value ===")
print(f"  Test Statistic (t) : {t_stat:.4f}")
print(f"  P-Value            : {p_val:.4f}")

# ── Final Business Conclusion ─────────────────────────────────────────
print("\n=== Final Business Conclusion ===")
if p_val < 0.05:
    direction = "above" if richmond.mean() > mu_0 else "below"
    print(f"  Decision : REJECT H0  (p = {p_val:.4f} < 0.05)")
    print(f"  The actual mean Richmond price (${richmond.mean():,.0f}) is statistically")
    print(f"  significantly {direction} the firm's claimed $1,000,000.")
    print("  Business Conclusion: The firm's pricing claim of $1,000,000 is NOT")
    print("  supported by the data. Valuations and investment models should use")
    print(f"  the empirically derived mean of ${richmond.mean():,.0f} instead.")
else:
    print(f"  Decision : FAIL TO REJECT H0  (p = {p_val:.4f} >= 0.05)")
    print("  The data is consistent with the firm's $1,000,000 claim.")
    print("  Business Conclusion: The pricing claim holds statistically.")

=== Descriptive Statistics — Richmond ===
  n       : 260
  Mean    : $1,083,564.42
  Median  : $1,078,000.00
  Std Dev : $522,353.52

=== Null and Alternative Hypotheses ===
  H0 (Null)       : mu = $1,000,000  (firm's claim is correct)
  H1 (Alternative): mu != $1,000,000  (actual mean is significantly different)
  Test type       : Two-tailed, alpha = 0.05

=== Test Statistic & P-Value ===
  Test Statistic (t) : 2.5795
  P-Value            : 0.0104

=== Final Business Conclusion ===
  Decision : REJECT H0  (p = 0.0104 < 0.05)
  The actual mean Richmond price ($1,083,564) is statistically
  significantly above the firm's claimed $1,000,000.
  Business Conclusion: The firm's pricing claim of $1,000,000 is NOT
  supported by the data. Valuations and investment models should use
  the empirically derived mean of $1,083,564 instead.


---
## Question 7: Independent Two-Sample T-Test — Car Parking Effect on Price
**Question:** Do properties with car parking sell at a higher average price than properties without car parking?  
**Significance level:** α = 0.05


In [8]:
# ── Split dataset by car parking ──────────────────────────────────────
car_df      = df[['Price', 'Car']].dropna()
with_car    = car_df[car_df['Car'] > 0]['Price']
without_car = car_df[car_df['Car'] == 0]['Price']

print("=== Descriptive Statistics ===")
print(f"  WITH car parking    : n={len(with_car):,},  Mean=${with_car.mean():,.2f},  Std=${with_car.std():,.2f}")
print(f"  WITHOUT car parking : n={len(without_car):,},  Mean=${without_car.mean():,.2f},  Std=${without_car.std():,.2f}")
print(f"  Observed difference : ${with_car.mean() - without_car.mean():,.2f}")

# ── Choice of Test ─────────────────────────────────────────────────────
print("\n=== Choice of Test ===")
print("  Test: Welch's Independent Two-Sample T-Test (one-tailed, right)")
print("  Justification:")
print("    - Two independent, mutually exclusive groups (with vs without car)")
print("    - Continuous outcome variable (Price)")
print("    - Unequal sample sizes => Welch's t-test is preferred over Student's")
print("      as it does not assume equal variances")
print("    - Directional claim (with car > without car) => one-tailed right test")

# ── Hypotheses ─────────────────────────────────────────────────────────
print("\n=== Hypotheses ===")
print("  H0: mu_with_car <= mu_without_car  (no premium for car parking)")
print("  H1: mu_with_car  > mu_without_car  (car parking commands higher price)")

# ── Test ───────────────────────────────────────────────────────────────
t_stat, p_two = ttest_ind(with_car, without_car, equal_var=False)
p_one = p_two / 2   # one-tailed right

print("\n=== Test Results ===")
print(f"  T-statistic        : {t_stat:.4f}")
print(f"  Two-tailed p-value : {p_two:.4f}")
print(f"  One-tailed p-value : {p_one:.4f}")

# ── P-Value Interpretation & Business Implications ─────────────────────
print("\n=== P-Value Interpretation ===")
print(f"  p = {p_one:.4f} > 0.05 => FAIL TO REJECT H0")
print("  Despite the intuitive expectation, the data does NOT provide")
print("  statistically significant evidence that car parking increases sale price.")
print("  The mean price difference is negligible (-$4,644 in favour of WITHOUT car).")

print("\n=== Business Implications for Developers ===")
print("  -> Car parking alone is NOT a statistically proven price driver")
print("     in this dataset when evaluated across all suburbs.")
print("  -> Developers should NOT assume a blanket price premium from car parking.")
print("  -> Other factors (suburb, property type, size) likely dominate pricing.")
print("  -> Suburb-level analysis is recommended before committing to car parking")
print("     as a development investment for price uplift.")

=== Descriptive Statistics ===
  WITH car parking    : n=12,492,  Mean=$1,074,443.92,  Std=$649,143.82
  WITHOUT car parking : n=1,026,  Mean=$1,079,088.01,  Std=$513,754.33
  Observed difference : $-4,644.09

=== Choice of Test ===
  Test: Welch's Independent Two-Sample T-Test (one-tailed, right)
  Justification:
    - Two independent, mutually exclusive groups (with vs without car)
    - Continuous outcome variable (Price)
    - Unequal sample sizes => Welch's t-test is preferred over Student's
      as it does not assume equal variances
    - Directional claim (with car > without car) => one-tailed right test

=== Hypotheses ===
  H0: mu_with_car <= mu_without_car  (no premium for car parking)
  H1: mu_with_car  > mu_without_car  (car parking commands higher price)

=== Test Results ===
  T-statistic        : -0.2722
  Two-tailed p-value : 0.7855
  One-tailed p-value : 0.3927

=== P-Value Interpretation ===
  p = 0.3927 > 0.05 => FAIL TO REJECT H0
  Despite the intuitive expectation

---
## Question 8: Two-Way ANOVA — Location & Property Type Effect on Price
**Task:** Investigate whether property prices are influenced by Suburb, Type of property, and their interaction.  
**Top 5 suburbs by volume used** for a robust, balanced ANOVA.


In [9]:
# ── Prepare data ──────────────────────────────────────────────────────
top_suburbs = df['Suburb'].value_counts().head(5).index.tolist()
anova_df    = df[df['Suburb'].isin(top_suburbs)][['Price', 'Suburb', 'Type']].dropna()

print("=== ANOVA Dataset ===")
print(f"  Suburbs used   : {top_suburbs}")
print(f"  Property types : {sorted(anova_df['Type'].unique())}  (h=house, t=townhouse, u=unit)")
print(f"  Total records  : {len(anova_df):,}")

print("\n=== Mean Price by Suburb ===")
print(anova_df.groupby('Suburb')['Price'].mean().apply(lambda x: f"${x:,.0f}").to_string())

print("\n=== Mean Price by Property Type ===")
type_map = {'h': 'House', 't': 'Townhouse', 'u': 'Unit/Apartment'}
means = anova_df.groupby('Type')['Price'].mean()
for t, m in means.items():
    print(f"  {type_map.get(t, t):15s}: ${m:,.0f}")

# ── Two-Way ANOVA ──────────────────────────────────────────────────────
model      = ols('Price ~ C(Suburb) + C(Type) + C(Suburb):C(Type)', data=anova_df).fit()
anova_tbl  = anova_lm(model, typ=2)

print("\n=== Two-Way ANOVA Table (Type II SS) ===")
print(anova_tbl.to_string())

# ── Interpretation ─────────────────────────────────────────────────────
alpha = 0.05
print("\n=== Factor Significance at alpha = 0.05 ===")
labels = {
    'C(Suburb)'          : 'Suburb (Main Effect)',
    'C(Type)'            : 'Property Type (Main Effect)',
    'C(Suburb):C(Type)'  : 'Suburb x Type (Interaction)',
}
for factor in anova_tbl.index:
    p = anova_tbl.loc[factor, 'PR(>F)']
    if hasattr(p, '__float__') and not (p != p):  # not NaN
        sig  = "SIGNIFICANT" if p < alpha else "Not significant"
        name = labels.get(factor, factor)
        print(f"  {name:40s}: p = {p:.2e}  =>  {sig}")

print("\n=== Business Conclusions ===")
print("  1. SUBURB significantly affects property price.")
print("     -> Location remains the dominant value driver in Australian real estate.")
print("     -> Pricing models must be suburb-specific, not market-wide.")
print("  2. PROPERTY TYPE significantly affects price.")
print("     -> Houses command the highest premiums; units/apartments the lowest.")
print("     -> Developers can use type as a pricing lever.")
print("  3. SUBURB x TYPE INTERACTION is also significant.")
print("     -> The price gap between houses and units varies by suburb.")
print("     -> Strategy must consider the suburb-type combination, not each factor alone.")

=== ANOVA Dataset ===
  Suburbs used   : ['Reservoir', 'Richmond', 'Bentleigh East', 'Preston', 'Brunswick']
  Property types : ['h', 't', 'u']  (h=house, t=townhouse, u=unit)
  Total records  : 1,329

=== Mean Price by Suburb ===
Suburb
Bentleigh East    $1,085,592
Brunswick         $1,013,171
Preston             $902,801
Reservoir           $690,009
Richmond          $1,083,564

=== Mean Price by Property Type ===
  House          : $1,057,637
  Townhouse      : $849,650
  Unit/Apartment : $536,466

=== Two-Way ANOVA Table (Type II SS) ===
                         sum_sq      df           F         PR(>F)
C(Suburb)          4.160952e+13     4.0  139.142810   3.337329e-99
C(Type)            6.480908e+13     2.0  433.444923  2.721411e-145
C(Suburb):C(Type)  6.129412e+12     8.0   10.248420   5.587511e-14
Residual           9.823523e+13  1314.0         NaN            NaN

=== Factor Significance at alpha = 0.05 ===
  Suburb (Main Effect)                    : p = 3.34e-99  =>  SIGNIFICAN

---
## Question 9: p-Value Interpretation — Decision Making
**Scenario:** A hypothesis test comparing property prices across two suburbs produces a **p-value of 0.032**.

---

### 1. What does this p-value indicate?
A p-value of **0.032** means there is only a **3.2% probability** of observing a price difference this large (or larger) purely by random chance, *if the null hypothesis were true* (i.e., if the two suburbs had the same mean price).  
This is strong evidence against H₀ — the observed difference is unlikely to be a statistical accident.

---

### 2. Should the null hypothesis be rejected at α = 0.05?
**Yes.**  
Since **p = 0.032 < α = 0.05**, the result is **statistically significant at the 5% level**.  
We **reject H₀** (that the two suburbs have equal mean prices).

---

### 3. How should a business stakeholder interpret this result?

| Item | Detail |
|---|---|
| **Statistical conclusion** | Prices differ significantly between the two suburbs |
| **Confidence** | 96.8% confidence the difference is real, not random |
| **Action** | Use suburb-specific pricing strategies and investment thresholds |
| **Caution** | Statistical significance ≠ practical significance — also evaluate the *magnitude* of the price gap (effect size) before committing capital |

> **Bottom line:** The two suburbs should be treated as distinct pricing markets. A one-size-fits-all valuation approach is not statistically defensible.


---
## Question 10: Industry-Style Hypothesis Validation — Premium for >2 Bathrooms
**Claim by housing policy group:** Properties with more than 2 bathrooms command a premium price.  
**Task:** Design and execute a statistical test; report p-value; give a clear recommendation.


In [10]:
# ── Prepare data ──────────────────────────────────────────────────────
bath_price  = df[['Price', 'Bathroom']].dropna()
more_2      = bath_price[bath_price['Bathroom'] > 2]['Price']
two_or_less = bath_price[bath_price['Bathroom'] <= 2]['Price']

print("=== Descriptive Statistics ===")
print(f"  Properties with >2 bathrooms  : n={len(more_2):,},  Mean=${more_2.mean():,.2f},  Median=${more_2.median():,.2f}")
print(f"  Properties with <=2 bathrooms : n={len(two_or_less):,},  Mean=${two_or_less.mean():,.2f},  Median=${two_or_less.median():,.2f}")
print(f"  Price premium (>2 bath)        : ${more_2.mean() - two_or_less.mean():,.2f}")

# ── Identify the Correct Test ──────────────────────────────────────────
print("\n=== Test Identification ===")
print("  Test: Welch's Independent Two-Sample T-Test (one-tailed, right)")
print("  Why Welch's:")
print("    - Two independent groups (>2 bath vs <=2 bath)")
print("    - Continuous numeric outcome variable (Price)")
print("    - Unequal and large sample sizes => Welch's handles unequal variances")
print("  Why one-tailed (right):")
print("    - The policy group's claim is directional: premium exists for >2 bath")
print("    - H1 specifies mu_(>2) > mu_(<=2), not just 'different'")

# ── State Hypotheses ───────────────────────────────────────────────────
print("\n=== Hypotheses ===")
print("  H0: mu_(>2 bath) <= mu_(<=2 bath)  (no bathroom premium — policy claim is false)")
print("  H1: mu_(>2 bath)  > mu_(<=2 bath)  (bathroom premium exists — policy claim is true)")

# ── Execute Test & Report P-Value ─────────────────────────────────────
t_stat, p_two = ttest_ind(more_2, two_or_less, equal_var=False)
p_one = p_two / 2

print("\n=== Test Results ===")
print(f"  T-statistic        : {t_stat:.4f}")
print(f"  Two-tailed p-value : {p_two:.2e}")
print(f"  One-tailed p-value : {p_one:.2e}")

# ── Recommendation ─────────────────────────────────────────────────────
print("\n=== Recommendation to Policymakers ===")
if p_one < 0.05 and t_stat > 0:
    print(f"  Decision : REJECT H0  (p << 0.05)")
    print(f"  The data VALIDATES the policy group's claim.")
    print(f"  Properties with >2 bathrooms command a mean premium of")
    print(f"  ${more_2.mean() - two_or_less.mean():,.0f} — a statistically significant finding.")
    print("")
    print("  Policy Recommendations:")
    print("  1. CLASSIFICATION: Use bathroom count (>2 vs <=2) as a legitimate")
    print("     tier boundary for affordable vs premium housing policy.")
    print("  2. STAMP DUTY / TAX: Consider higher property tax brackets for")
    print("     homes with >2 bathrooms, reflecting their higher market value.")
    print("  3. GRANTS & SUBSIDIES: Target housing grants to <=2 bath properties")
    print("     to ensure affordability support reaches the right segment.")
    print("  4. DEVELOPER INCENTIVES: Reward developments that cap at 2 bathrooms")
    print("     to keep supply in the affordable segment.")
else:
    print("  Decision : FAIL TO REJECT H0")
    print("  The data does not support the policy group's claim.")

=== Descriptive Statistics ===
  Properties with >2 bathrooms  : n=1,060,  Mean=$1,882,824.20,  Median=$1,661,500.00
  Properties with <=2 bathrooms : n=12,520,  Mean=$1,007,347.94,  Median=$870,000.00
  Price premium (>2 bath)        : $875,476.26

=== Test Identification ===
  Test: Welch's Independent Two-Sample T-Test (one-tailed, right)
  Why Welch's:
    - Two independent groups (>2 bath vs <=2 bath)
    - Continuous numeric outcome variable (Price)
    - Unequal and large sample sizes => Welch's handles unequal variances
  Why one-tailed (right):
    - The policy group's claim is directional: premium exists for >2 bath
    - H1 specifies mu_(>2) > mu_(<=2), not just 'different'

=== Hypotheses ===
  H0: mu_(>2 bath) <= mu_(<=2 bath)  (no bathroom premium — policy claim is false)
  H1: mu_(>2 bath)  > mu_(<=2 bath)  (bathroom premium exists — policy claim is true)

=== Test Results ===
  T-statistic        : 28.8002
  Two-tailed p-value : 7.38e-137
  One-tailed p-value : 3.69e-13